In [1]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio 
from scipy.fft import fft, fftfreq
from scipy import signal
from scipy.signal import filtfilt ,freqz, firwin, freqz

In [ ]:
# Load the audio file
audio_file = './dataset/train/Drone/5m-50m-ex6.wav'  # Replace with your audio file path
#audio_file = './dataset/train/No_Drone/เสียงธรรมชาติ(New_Sound)-ex19.wav'
#audio_file = './dataset/train/No_Drone/เสียงธรรมชาติ(New_Sound)-ex12.wav'
y, sr = librosa.load(audio_file)  # , duration=20)

y = y[:22000]
timesDuration = librosa.get_duration(y=y, sr=sr)

# normalize audio  
max_value = np.max(np.abs(y))       # Determine the maximum values
audio_normalize = y/max_value        # Use max_value and normalize sound data to get values between -1 & +1

print(f'Sampling Rate: {sr} Hz')
print(f'Audio Duration: {timesDuration:.0f} seconds')

In [ ]:
Audio(data=audio_normalize, rate=sr)

# Power Spectrum Density

In [ ]:
# Set the parameters for the PSD
NFFT = 2048
noverlap = NFFT // 2

# Compute the PSD of the audio signal
f, psd = signal.welch(audio_normalize, fs=sr, nperseg=NFFT, noverlap=noverlap)

# Plot the PSD
plt.figure(figsize=(10, 4))
plt.plot(f, 10*np.log10(psd))
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power Spectral Density (dB/Hz)')
plt.show()

# Identify the noise frequencies
threshold = np.mean(psd) + np.std(psd)
noise_freqs = f[psd > threshold]
#print('Noise frequencies:', noise_freqs)

In [5]:
# Calculate Spectrogram by using SFTF method
def spectrogram_cal(data):
    n_fft = 2048       # Length of FFT window
    hop_length = 32   # Number of samples between frames
    win_length = 1024  # Length of the window
    window = 'hann'    # Windowing function

    # Compute the STFT
    spectrogram = librosa.stft(data, n_fft=n_fft, hop_length=hop_length, win_length=win_length, window=window)
    spectrogram_db = librosa.amplitude_to_db(np.abs(spectrogram))  # Convert to dB
    
    return spectrogram_db

In [ ]:
# Plot the power spectrum

spectrogram_dB = spectrogram_cal(audio_normalize)
print("Shape of spectrogram :" + str(spectrogram_dB.shape))

plt.figure(figsize=(10, 8))
plt.subplot(2, 1, 1)
plt.title(f'Spectrogram')
librosa.display.specshow(spectrogram_dB, sr=sr, x_axis='time', y_axis='log', cmap='viridis')
#cmap = 'viridis', 'plasma', 'inferno', 'magma', 'cividis'
plt.colorbar(label='Power (dB)')
#plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram (STFT configured)')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.grid()




# Filter design, visualize and save filtered signal

In [ ]:

# Set the bandpass filter parameters
low_freq = 430  # Hz
high_freq = 624  # Hz
filter_order = 4

# Create the bandpass filter
nyquist_freq = 0.5 * sr
low = low_freq / nyquist_freq
high = high_freq / nyquist_freq
b, a = signal.butter(filter_order, [low, high], btype='band')

# Apply the filter to remove noise from the audio signal
audio_filtered = signal.filtfilt(b, a, audio_normalize)

# Plot the original and filtered audio signals
plt.figure(figsize=(10, 4))
plt.plot(np.arange(len(audio_normalize)) / sr, audio_normalize, label='Original')
plt.plot(np.arange(len(audio_filtered)) / sr, audio_filtered, label='Filtered')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.show()


In [ ]:
Audio(data=audio_normalize, rate=sr)

In [ ]:
Audio(data=audio_filtered, rate=sr)